To return plots generated by `opti_multi_pump_clean.py`, switch the file names found in the folder 'Output' and run every cell in this notebook. The plots should pop out and be interactive.

In [13]:
# Outputs
file_path = r'..\src\Output\2_retrofit_electric_20260130_185807_data_output.xlsx'
pkl_file_path = r'..\src\Output\2_retrofit_electric_20260130_185807_pipe_data_output.pkl'

# Data (Gogma)
data_file = r'..\..\DO_NOT_DISTRIBUTE\DO_NOT_DISTRIBUTE_DATA_GOGMA.xlsx'

In [14]:
import pandas as pd
import numpy as np
import pickle
import networkx as nx
%matplotlib tk
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from geopy.distance import great_circle
from matplotlib.lines import Line2D
from opti_multi_pump_clean import InteractiveParetoPlot

In [15]:
def recover_data(file_path, pkl_file_path):
    data_dict = pd.read_excel(file_path, sheet_name=None)
    all_positions = []
    all_values = []
    all_indices = []
    all_X = []
    max_nb_fountains = len(data_dict) - 1 
    initial_impact = data_dict['data_summary']['initial_impact'][0]
    
    counter = 0
    for sheet_name, df in data_dict.items():
        if sheet_name != 'data_summary':
            all_positions.append(df.iloc[:,counter:-2].to_numpy())
            all_values.append(df.iloc[:,-2:].to_numpy())
            all_indices.append(df.iloc[:,:counter].to_numpy())
            all_X.append(df.iloc[:,:-2].to_numpy())
        counter += 1

    pipe_data = pickle.load(open(pkl_file_path,'rb'))
    return all_positions, all_values, all_indices, all_X, max_nb_fountains, initial_impact, pipe_data

In [16]:
def read_data_gogma(file_path):
    data_households, data_sources = pd.read_excel(
        file_path,
        sheet_name=['Position surveyed household (hh','Position water sources'],
        header=None).values()
    data_sources = data_sources.iloc[:,:5]
    data_sources.columns = ['ID','Name','Lon','Lat','Altitude']
    data_sources['Type'] = data_sources['Name'].str[1]
    data_sources = data_sources[(data_sources['Type']=='B') | (data_sources['Type']=='W')].drop('Name',axis=1)
    data_sources['Type'] = data_sources['Type'].replace('B','Borehole')
    data_sources['Type'] = data_sources['Type'].replace('W','Open Well')
    
    data_households.columns = ['ID','Lat','Lon','Altitude']
    data_household_capita = pd.read_excel(file_path,sheet_name='Choice source before install',usecols='A,F,G').iloc[1:]
    data_household_capita.columns = ['ID','Reading','Nb capita']
    data_household_capita = data_household_capita.sort_values(by='Reading',ascending=True)
    data_household_capita = data_household_capita.drop_duplicates(subset='ID').sort_values(by='ID').iloc[:-1].drop('Reading',axis=1)
    
    data_households['Type'] = 'Household'
    data_households = pd.merge(data_households, data_household_capita, on='ID', how='inner')    
    
    data = pd.concat([data_households,data_sources],ignore_index=True)
    households = data_households
    pumps = data_sources[data_sources['Type']=='Borehole']
    open_wells = data_sources[data_sources['Type']=='Open Well']
    
    
    return data,households,pumps,open_wells

In [17]:
# Import data and define arrays (may need to rewrite read function depending on input)
data,households,pumps,open_wells = read_data_gogma(data_file)
pos_households = households[['Lon','Lat']].to_numpy() 
nb_capita = households['Nb capita'].to_numpy()
pos_pumps = pumps[['Lon','Lat']].to_numpy()
bounds = np.array([
    [data['Lon'].min(), data['Lat'].min()],  # Min bounds
    [data['Lon'].max(), data['Lat'].max()],  # Max bounds
])

# Create gridpoints for relief plot
grid_x, grid_y = np.mgrid[data['Lon'].min():data['Lon'].max():200j, 
                        data['Lat'].min():data['Lat'].max():200j]
grid_z = griddata((data['Lon'], data['Lat']), data['Altitude'], (grid_x, grid_y), method='cubic')

In [18]:
all_positions, all_result_vals, all_indices, all_X, max_nb_fountains, initial_impact, pipe_data = recover_data(file_path, pkl_file_path)
plotter = InteractiveParetoPlot(all_positions, all_result_vals, all_indices, 
                      households, pos_pumps, grid_x, grid_y, grid_z, initial_impact, pipe_data, "impact",
                      max_nb_fountains, all_X)
plotter.show()

### Comparing two pareto fronts

In [7]:
# Outputs
file_path = [r'..\src\Output\1_retrofit_electric_20260130_182849_data_output.xlsx',
             r'..\src\Output\2_retrofit_electric_20260130_185807_data_output.xlsx',
             ]
pkl_file_path = [r'..\src\Output\1_retrofit_electric_20260130_182849_pipe_data_output.pkl',
                 r'..\src\Output\2_retrofit_electric_20260130_185807_pipe_data_output.pkl',
                 ]

In [8]:
result_vals = []
initial_imps = []
for excel,pkl_file in zip(file_path,pkl_file_path):
    all_positions, all_result_vals, all_indices, all_X, max_nb_fountains, initial_impact, pipe_data = recover_data(excel, pkl_file)
    result_vals.append(all_result_vals)
    initial_imps.append(initial_impact)

In [11]:
colours = ['green','red']
shapes = ['.','o','v']

In [12]:
fig,ax = plt.subplots(figsize =  (12,7))
for i,results in enumerate(result_vals):
    initial_impact_val = initial_imps[i]
    for j,solutions in enumerate(results):
        label = f'n_retrofit = {i+1}, n_fountains = {j}'
        ax.scatter(solutions[:,0]-initial_impact_val,solutions[:,1],label=label, marker=shapes[j],color=colours[i])

ax.set_xlabel("Impact (person-meters $*10^3$)")
ax.set_ylabel("Cost (€)")
ax.legend()
ax.grid()
fig.savefig("pareto_comparison.svg")
plt.show()